# Lab 01: Hugging Face Transformers - Text Classification & NER

**Learning Objectives:**
- Load pre-trained transformer models
- Perform text classification
- Named entity recognition (NER)
- Sentiment analysis
- Compare different model architectures

**Prerequisites:**
```bash
pip install transformers torch datasets
```

In [ ]:
# Install required packages
!pip install -q transformers torch datasets

## Part 1: Text Classification with BERT

In [ ]:
from transformers import pipeline

# Load sentiment analysis pipeline
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

# Test examples
texts = [
    "I absolutely loved this product! It exceeded my expectations.",
    "This is the worst purchase I've ever made. Very disappointed.",
    "The product is okay, nothing special but works as intended."
]

results = classifier(texts)

for text, result in zip(texts, results):
    print(f"Text: {text}")
    print(f"Sentiment: {result['label']}, Confidence: {result['score']:.4f}\n")

## Part 2: Manual Model Loading and Tokenization

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Load model and tokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Tokenize input
text = "Transformers are revolutionizing natural language processing!"
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)

print("Input IDs:", inputs["input_ids"])
print("Attention Mask:", inputs["attention_mask"])
print("\nDecoded tokens:")
print(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))

## Part 3: Named Entity Recognition (NER)

In [ ]:
# Load NER pipeline
ner = pipeline("ner", model="dslim/bert-base-NER", grouped_entities=True)

# Example text
text = """
Apple Inc. was founded by Steve Jobs, Steve Wozniak, and Ronald Wayne in April 1976.
The company is headquartered in Cupertino, California. In 2023, Tim Cook served as CEO.
"""

# Extract entities
entities = ner(text)

print("Named Entities Found:\n")
for entity in entities:
    print(f"Entity: {entity['word']}")
    print(f"Type: {entity['entity_group']}")
    print(f"Score: {entity['score']:.4f}\n")

## Part 4: Question Answering

In [ ]:
# Load Q&A pipeline
qa_pipeline = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")

# Context and questions
context = """
The transformer architecture was introduced in the paper 'Attention is All You Need' 
by Vaswani et al. in 2017. It uses self-attention mechanisms to process sequences in 
parallel, unlike RNNs which process sequentially. This innovation enabled training much 
larger models and led to breakthroughs like BERT and GPT.
"""

questions = [
    "When was the transformer architecture introduced?",
    "What mechanism does the transformer use?",
    "Who introduced the transformer?"
]

for question in questions:
    result = qa_pipeline(question=question, context=context)
    print(f"Q: {question}")
    print(f"A: {result['answer']} (confidence: {result['score']:.4f})\n")

## Part 5: Zero-Shot Classification

In [ ]:
# Load zero-shot classification pipeline
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Text to classify
text = "I'm looking for a new laptop with good battery life and a lightweight design."

# Candidate labels (no training needed!)
candidate_labels = ["technology", "sports", "politics", "entertainment", "shopping"]

result = classifier(text, candidate_labels)

print("Text:", text)
print("\nClassification Results:")
for label, score in zip(result['labels'], result['scores']):
    print(f"{label}: {score:.4f}")

## Part 6: Batch Processing for Efficiency

In [ ]:
import time

# Create test dataset
texts = [
    "This movie was absolutely fantastic!",
    "I hated every minute of it.",
    "Pretty average, nothing special.",
    "Best film I've seen this year!",
    "Waste of time and money."
] * 20  # 100 examples

# Single processing
start = time.time()
for text in texts:
    _ = classifier(text)
single_time = time.time() - start

# Batch processing
start = time.time()
_ = classifier(texts, batch_size=32)
batch_time = time.time() - start

print(f"Single processing: {single_time:.2f}s")
print(f"Batch processing: {batch_time:.2f}s")
print(f"Speedup: {single_time/batch_time:.2f}x")

## Part 7: Model Comparison

In [ ]:
# Compare different models on same task
models = [
    "distilbert-base-uncased-finetuned-sst-2-english",
    "cardiffnlp/twitter-roberta-base-sentiment",
]

test_text = "The new iPhone is amazing but way too expensive."

print(f"Text: {test_text}\n")

for model_name in models:
    print(f"Model: {model_name}")
    clf = pipeline("sentiment-analysis", model=model_name)
    result = clf(test_text)[0]
    print(f"Result: {result['label']} ({result['score']:.4f})\n")

## Part 8: Feature Extraction (Embeddings)

In [ ]:
from transformers import AutoModel
import torch

# Load BERT for embeddings
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Get embeddings
text = "Transformers create rich contextual embeddings."
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
    # Use [CLS] token embedding
    cls_embedding = outputs.last_hidden_state[:, 0, :]
    # Or use mean pooling
    mean_embedding = outputs.last_hidden_state.mean(dim=1)

print(f"CLS embedding shape: {cls_embedding.shape}")
print(f"Mean embedding shape: {mean_embedding.shape}")
print(f"\nFirst 10 dimensions of CLS embedding:")
print(cls_embedding[0, :10])

## Exercises

1. **Custom Dataset:** Use the `datasets` library to load a dataset and perform classification
2. **Multi-label Classification:** Modify the code to classify text into multiple categories
3. **Attention Visualization:** Extract and visualize attention weights
4. **Error Analysis:** Find examples where the model fails and analyze why
5. **Model Comparison:** Test 5 different models and compare performance

## Additional Resources

- [Hugging Face Documentation](https://huggingface.co/docs/transformers)
- [Model Hub](https://huggingface.co/models)
- [Datasets Hub](https://huggingface.co/datasets)
- [Transformers Course](https://huggingface.co/course)